# ⏩ Sequenzielle Agenten-Workflows mit Microsoft Foundry (Python)

## 📋 Fortgeschrittenes Tutorial zur sequentiellen Verarbeitung

Dieses Notebook demonstriert **sequentielle Workflow-Muster** mit dem Microsoft Agent Framework. Sie lernen, wie man ausgeklügelte mehrstufige Verarbeitungspipelines erstellt, bei denen Agenten in einer bestimmten Reihenfolge ausgeführt werden und Daten und Kontext zwischen den Phasen weitergeben.

> **Migrationshinweis:** Dieses Beispiel verwendete zuvor GitHub Models. GitHub Models wird eingestellt (im Juli 2026), daher wird nun **Microsoft Foundry** über den `FoundryChatClient` verwendet, der auf die Azure OpenAI **Responses API** zugreift.

## 🎯 Lernziele

### 🔄 **Muster der sequentiellen Verarbeitung**
- **Lineares Workflow-Design**: Erstellung von schrittweisen Verarbeitungspipelines
- **Datenflusssteuerung**: Informationsweitergabe zwischen sequentiellen Agenten
- **Stage-Gate-Verarbeitung**: Implementierung von Kontrollpunkten und Validierungsphasen
- **Fortschrittsverfolgung**: Überwachung der Workflow-Ausführung und Zwischenresultate

### 🏗️ **Architektur von Unternehmenspipelines**
- **Modellierung von Geschäftsprozessen**: Abbildung realer Geschäftsprozesse auf Agenten-Workflows
- **Qualitätssicherung**: Mehrstufige Validierungs- und Prüfprozesse
- **Dokumentenverarbeitung**: Sequentielle Dokumentenanalyse und -transformation
- **Inhaltsproduktion**: Redaktionelle Workflows mit Prüf- und Freigabestufen

### 📊 **Fortgeschrittene Workflow-Funktionen**
- **Kontextbewahrung**: Erhaltung des Zustands über Workflow-Phasen hinweg
- **Fehlerweitergabe**: Umgang mit Ausfällen in der sequentiellen Verarbeitung
- **Leistungsoptimierung**: Effiziente Muster für die sequentielle Ausführung
- **Prüfpfade**: Vollständige Nachverfolgung sequentieller Operationen

## ⚙️ Voraussetzungen & Einrichtung

### 📦 **Abhängigkeiten**
```bash
pip install agent-framework -U
```

### 🔑 **Konfiguration**

Melden Sie sich mit der Azure CLI (`az login`) an, damit `AzureCliCredential` sich authentifizieren kann, und legen Sie dann Ihre Microsoft Foundry-Projektdaten fest.

```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-4.1-mini
```

## 🏢 **Anwendungsfälle für unternehmensweite sequentielle Workflows**

### 📝 **Dokumentenverarbeitungspipeline**
```
Raw Document → Content Extraction → Analysis → Validation → Final Output
```

### 🔍 **Workflow zur Qualitätssicherung** 
```
Initial Review → Technical Validation → Compliance Check → Final Approval
```

### 📰 **Pipeline zur Inhaltsproduktion**
```
Research → Writing → Editing → Review → Publishing
```

### 💼 **Automatisierung von Geschäftsprozessen**
```
Data Collection → Processing → Analysis → Report Generation → Distribution
```

## 🎨 **Prinzipien des sequentiellen Workflow-Designs**

- **🔗 Lineare Progression**: Jede Phase hängt vom Ergebnis der vorherigen Phase ab
- **📋 Zustandsverwaltung**: Kontext und Daten über alle Phasen bewahren
- **🛡️ Fehlerbehandlung**: Umgang mit Ausfällen in jeder Phase auf elegante Weise
- **📊 Fortschrittsüberwachung**: Verfolgung von Abschluss und Leistung in jeder Phase
- **🔄 Wiederverwendbarkeit von Phasen**: Gestaltung wiederverwendbarer Workflow-Komponenten

Lassen Sie uns ausgeklügelte sequentielle Verarbeitungsworkflows erstellen! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
from agent_framework import (
    Message,
    WorkflowBuilder,
    WorkflowEvent,
    WorkflowViz,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:

import os
import base64
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
SalesAgentName = "Sales-Agent"
SalesAgentInstructions = "You are my furniture sales consultant, you can find different furniture elements from the pictures and give me a purchase suggestion"

In [ ]:
PriceAgentName = "Price-Agent"
PriceAgentInstructions = """You are a furniture pricing specialist and budget consultant. Your responsibilities include:
        1. Analyze furniture items and provide realistic price ranges based on quality, brand, and market standards
        2. Break down pricing by individual furniture pieces
        3. Provide budget-friendly alternatives and premium options
        4. Consider different price tiers (budget, mid-range, premium)
        5. Include estimated total costs for room setups
        6. Suggest where to find the best deals and shopping recommendations
        7. Factor in additional costs like delivery, assembly, and accessories
        8. Provide seasonal pricing insights and best times to buy
        Always format your response with clear price breakdowns and explanations for the pricing rationale."""

In [ ]:
QuoteAgentName = "Quote-Agent"
QuoteAgentInstructions = """You are a assistant that create a quote for furniture purchase.
        1. Create a well-structured quote document that includes:
        2. A title page with the document title, date, and client name
        3. An introduction summarizing the purpose of the document
        4. A summary section with total estimated costs and recommendations
        5. Use clear headings, bullet points, and tables for easy readability
        6. All quotes are presented in markdown form"""

In [ ]:
sales_agent = provider.as_agent(
    name=SalesAgentName,
    instructions=SalesAgentInstructions,
)

price_agent = provider.as_agent(
    name=PriceAgentName,
    instructions=PriceAgentInstructions,
)

quote_agent = provider.as_agent(
    name=QuoteAgentName,
    instructions=QuoteAgentInstructions,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=sales_agent)
    .add_edge(sales_agent, price_agent)
    .add_edge(price_agent, quote_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra (`pip install graphviz`) plus the
# graphviz system binary; if it's not available, fall back to the text strings above.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
image_path = "../imgs/home.png"
with open(image_path, "rb") as image_file:
    image_b64 = base64.b64encode(image_file.read()).decode()
image_uri = f"data:image/png;base64,{image_b64}"


In [ ]:
# Note: the original notebook used a multimodal ChatMessage with an image of a
# living room. The current Message class no longer ships TextContent/DataContent
# helpers, so this migration uses a textual description of the same scene to
# keep the lesson focused on sequential workflow mechanics.
message = Message(
    role="user",
    text=(
        "I am furnishing a modern living room and want pieces that fit a warm, "
        "inviting style: a comfortable three-seat sofa, two accent armchairs, a "
        "wooden coffee table, a TV stand, a floor lamp, and a soft area rug. "
        "Please find appropriate furniture and give the corresponding price for "
        "each piece, then produce a final purchase quote."
    ),
)

In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The final stage (quote_agent) is the only output here.
events = await workflow.run(message)
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
